# Exploratory Data Analysis - Product Performance & Profitability
**Siswa 3 | Final Project Topik 7 | Hari Rabu - Minggu 1**

Notebook ini berisi analisis EDA terhadap dataset Sample Superstore, mencakup Univariate, Bivariate, Trend, dan Segmentation Analysis, untuk menjawab business question:

> **Produk/kategori mana yang paling menguntungkan dan paling merugikan perusahaan, dan apa faktor utama yang mempengaruhi profitabilitas produk?**

> ⚠️ **Catatan untuk Google Colab:** Sebelum menjalankan notebook ini, upload file `SuperStore.csv` ke Colab (klik ikon folder di sidebar kiri → tombol upload), atau mount Google Drive jika file disimpan di sana.

> 🎨 **Ketentuan Warna:** Di seluruh notebook ini, warna **hijau** menandakan performa tinggi / naik / di atas rata-rata perusahaan, sedangkan warna **merah** menandakan performa rendah / turun / di bawah rata-rata perusahaan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

df = pd.read_csv('SuperStore.csv')
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['YearMonth'] = df['Order Date'].dt.to_period('M').astype(str)
df.head()

## 1. Gambaran Umum Dataset

Dataset berisi **9.994 transaksi** penjualan retail (2015-2018), dengan kolom kunci: Sales, Profit, Discount, Quantity, Category, Sub-Category, Region, Segment, dan Ship Mode.

In [ ]:
print('Ukuran dataset:', df.shape)
print('\nMissing values:')
print(df.isnull().sum().sum(), 'total missing values')
print('\nStatistik deskriptif:')
df[['Sales','Profit','Discount','Quantity']].describe()

## 2. Univariate Analysis

### Visualisasi 1: Histogram - Distribusi Sales per Transaksi

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(df[df['Sales']<1000]['Sales'], bins=40, color='#2E6F9E', edgecolor='white')
ax.set_title('Distribusi Sales per Transaksi (< $1000)', fontsize=13, fontweight='bold')
ax.set_xlabel('Sales ($)'); ax.set_ylabel('Jumlah Transaksi')
plt.tight_layout()
plt.show()

**Insight 1:** Distribusi sales menceng ke kanan (right-skewed) - mayoritas transaksi bernilai kecil (di bawah $100), dengan sedikit transaksi bernilai besar yang menarik rata-rata ke atas.

### Visualisasi 2: Bar Chart - Jumlah Transaksi per Kategori

In [ ]:
cat_counts = df['Category'].value_counts()
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(cat_counts.index, cat_counts.values, color=['#2E6F9E','#E8834B','#4FA37B'])
ax.set_title('Jumlah Transaksi per Kategori', fontsize=13, fontweight='bold')
ax.set_ylabel('Jumlah Transaksi')
plt.tight_layout()
plt.show()

**Insight 2:** Office Supplies mendominasi jumlah transaksi (6.026, 60%), jauh di atas Furniture (2.121) dan Technology (1.847) - namun jumlah transaksi tinggi tidak otomatis berarti profit tinggi (akan dibahas di segmentasi produk).

### Visualisasi 3: Pie Chart - Proporsi Transaksi Untung vs Rugi

In [ ]:
untung = (df['Profit'] > 0).sum()
rugi = (df['Profit'] < 0).sum()
impas = (df['Profit'] == 0).sum()

labels = ['Untung', 'Rugi', 'Impas']
sizes = [untung, rugi, impas]
colors = ['#4FA37B', '#C9564A', '#B0B0B0']

fig, ax = plt.subplots(figsize=(7,6))
wedges, texts, autotexts = ax.pie(sizes, labels=labels, colors=colors,
                                    autopct=lambda p: f'{p:.1f}%' if p>2 else '',
                                    startangle=90, textprops={'fontsize':13, 'fontweight':'bold'},
                                    wedgeprops={'edgecolor':'white','linewidth':2})
for at in autotexts:
    at.set_color('white'); at.set_fontsize(14)
ax.set_title('Proporsi Transaksi Untung vs Rugi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Untung: {untung} ({untung/len(df)*100:.1f}%)')
print(f'Rugi: {rugi} ({rugi/len(df)*100:.1f}%)')

**Insight 3:** 80,6% transaksi (8.058) menghasilkan untung, sementara 18,7% (1.871) merugi - mayoritas transaksi tetap menguntungkan, namun hampir 1 dari 5 transaksi berpotensi menggerus margin keseluruhan perusahaan.

## 3. Bivariate Analysis

### Visualisasi 4: Bar Chart - Rata-rata Profit per Rentang Sales

In [ ]:
bins_sales = [0,50,100,250,500,1000,df['Sales'].max()+1]
labels_sales = ['<$50','$50-100','$100-250','$250-500','$500-1000','>$1000']
df['SalesBin'] = pd.cut(df['Sales'], bins=bins_sales, labels=labels_sales, include_lowest=True)
avg_profit_sales = df.groupby('SalesBin', observed=True)['Profit'].mean()

colors_bar = ['#4FA37B' if v>=0 else '#C9564A' for v in avg_profit_sales.values]
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(avg_profit_sales.index.astype(str), avg_profit_sales.values, color=colors_bar)
ax.axhline(0, color='#999999', linewidth=0.8)
ax.set_title('Rata-rata Profit per Rentang Sales', fontsize=13, fontweight='bold')
ax.set_xlabel('Rentang Sales'); ax.set_ylabel('Rata-rata Profit ($)')
plt.tight_layout()
plt.show()

print('Korelasi Sales-Profit:', df['Sales'].corr(df['Profit']).round(3))

**Insight 4:** Rata-rata profit naik konsisten seiring rentang sales makin tinggi - dari $3,7 per transaksi di bawah $50 menjadi $314 di atas $1000. Ini sejalan dengan korelasi positif moderat (r=0,48): sales besar cenderung menghasilkan profit lebih besar.

### Visualisasi 5: Bar Chart - Rata-rata Profit per Rentang Discount

In [ ]:
bins_disc = [-0.01,0.001,0.1,0.2,0.3,0.4,0.5,1.0]
labels_disc = ['0%','1-10%','11-20%','21-30%','31-40%','41-50%','>50%']
df['DiscBin'] = pd.cut(df['Discount'], bins=bins_disc, labels=labels_disc)
avg_profit_disc = df.groupby('DiscBin', observed=True)['Profit'].mean()

colors_bar2 = ['#4FA37B' if v>=0 else '#C9564A' for v in avg_profit_disc.values]
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(avg_profit_disc.index.astype(str), avg_profit_disc.values, color=colors_bar2)
ax.axhline(0, color='#999999', linewidth=0.8)
ax.set_title('Rata-rata Profit per Rentang Discount', fontsize=13, fontweight='bold')
ax.set_xlabel('Rentang Discount'); ax.set_ylabel('Rata-rata Profit ($)')
plt.tight_layout()
plt.show()

print('Korelasi Discount-Profit:', df['Discount'].corr(df['Profit']).round(3))

**Insight 5:** Rata-rata profit positif saat diskon di bawah 20% (tertinggi $96 pada diskon 1-10%), lalu berbalik negatif begitu diskon melewati 20% dan terus memburuk hingga -$299 pada diskon 41-50%. Ini menandakan diskon di atas 20% adalah titik kritis yang perlu dievaluasi ulang oleh perusahaan.

## 4. Trend Analysis

### Visualisasi 6: Line Chart - Trend Sales Bulanan (Hijau = naik, Merah = turun)

In [ ]:
monthly = df.groupby('YearMonth')['Sales'].sum()
vals = monthly.values

fig, ax = plt.subplots(figsize=(10,5))
for i in range(len(vals)-1):
    color = '#4FA37B' if vals[i+1] >= vals[i] else '#C9564A'
    ax.plot([i, i+1], [vals[i], vals[i+1]], color=color, linewidth=2.2)
ax.scatter(range(len(vals)), vals, color='#2E6F9E', s=14, zorder=5)
ax.set_title('Trend Sales Bulanan (2015-2018)\n(Hijau = naik dari bulan sebelumnya, Merah = turun)', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Sales ($)'); ax.set_xlabel('Bulan (Jan 2015 - Des 2018)')
ax.set_xticks(range(0, len(monthly), 6))
ax.set_xticklabels(monthly.index[::6], rotation=45)

from matplotlib.lines import Line2D
legend_elems = [Line2D([0],[0], color='#4FA37B', lw=2, label='Naik'), Line2D([0],[0], color='#C9564A', lw=2, label='Turun')]
ax.legend(handles=legend_elems, loc='upper left', frameon=False)
plt.tight_layout()
plt.show()

**Insight 6:** Sales menunjukkan pola musiman jelas - segmen hijau (naik) konsisten muncul menjelang November-Desember tiap tahun (kemungkinan akhir tahun/liburan), sementara segmen merah (turun) sering terjadi di awal tahun (Januari-Februari). Tren keseluruhan naik dari 2015 ke 2018 meski fluktuatif tiap bulan.

## 5. Segmentation Analysis

### Visualisasi 7: Segmentasi Produk - Profit per Sub-Category (Hijau = untung, Merah = rugi)

In [ ]:
sub = df.groupby('Sub-Category')['Profit'].sum().sort_values()
colors_sub = ['#C9564A' if v<0 else '#4FA37B' for v in sub.values]
fig, ax = plt.subplots(figsize=(9,6.5))
ax.barh(sub.index, sub.values, color=colors_sub)
ax.axvline(0, color='#999999', linewidth=0.8)
ax.set_title('Segmentasi Produk: Profit per Sub-Category\n(Hijau = untung, Merah = rugi)', fontsize=12, fontweight='bold')
ax.set_xlabel('Total Profit ($)')
plt.tight_layout()
plt.show()

**Insight 7:** 3 dari 17 sub-kategori merugi (merah): Tables (-$17.725), Bookcases (-$3.473), Supplies (-$1.189). Tables adalah kerugian terbesar dan jauh melampaui yang lain.

### Visualisasi 8: Segmentasi Customer - Margin Profit per Segment

In [ ]:
overall_margin = df['Profit'].sum()/df['Sales'].sum()*100

seg = df.groupby('Segment').agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
seg['Margin'] = seg['Profit']/seg['Sales']*100
seg = seg.sort_values('Margin')
colors_seg = ['#4FA37B' if m>=overall_margin else '#C9564A' for m in seg['Margin']]

fig, ax = plt.subplots(figsize=(8,5.5))
bars = ax.bar(seg.index, seg['Margin'], color=colors_seg)
ax.axhline(overall_margin, color='#555555', linewidth=1, linestyle='--')
ax.text(len(seg)-0.5, overall_margin+0.3, f'Rata-rata perusahaan: {overall_margin:.1f}%', ha='right', fontsize=9, color='#555555')
for b, m in zip(bars, seg['Margin']):
    ax.text(b.get_x()+b.get_width()/2, m+0.2, f'{m:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Segmentasi Customer: Margin Profit per Segment\n(Hijau = di atas rata-rata, Merah = di bawah rata-rata)', fontsize=12, fontweight='bold')
ax.set_ylabel('Margin Profit (%)')
plt.tight_layout()
plt.show()

print(seg)

**Insight 8:** Consumer (merah) margin-nya 11,5%, di bawah rata-rata perusahaan 12,5%, meski sales-nya paling besar ($1,16 juta). Corporate dan Home Office (hijau) justru lebih efisien dengan margin 13,0% dan 14,0% - ada trade-off antara volume dan efisiensi profit.

### Visualisasi 9: Segmentasi Wilayah - Margin Profit per Region

In [ ]:
reg = df.groupby('Region').agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
reg['Margin'] = reg['Profit']/reg['Sales']*100
reg = reg.sort_values('Margin')
colors_reg = ['#4FA37B' if m>=overall_margin else '#C9564A' for m in reg['Margin']]

fig, ax = plt.subplots(figsize=(8,5))
bars = ax.bar(reg.index, reg['Margin'], color=colors_reg)
ax.axhline(overall_margin, color='#555555', linewidth=1, linestyle='--')
ax.text(len(reg)-0.5, overall_margin+0.3, f'Rata-rata perusahaan: {overall_margin:.1f}%', ha='right', fontsize=9, color='#555555')
for b, m in zip(bars, reg['Margin']):
    ax.text(b.get_x()+b.get_width()/2, m+0.2, f'{m:.1f}%', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Segmentasi Wilayah: Margin Profit per Region\n(Hijau = di atas rata-rata, Merah = di bawah rata-rata)', fontsize=12, fontweight='bold')
ax.set_ylabel('Margin Profit (%)')
plt.tight_layout()
plt.show()

print(reg)

**Insight 9:** Central dan South Region (merah) berada di bawah rata-rata perusahaan - Central paling rendah (7,9%), hampir setengah dari West Region (hijau, 14,9%) yang menjadi wilayah paling efisien.

### Visualisasi 10: Segmentasi Channel - Margin Profit per Ship Mode

In [ ]:
ship = df.groupby('Ship Mode').agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
ship['Margin'] = ship['Profit']/ship['Sales']*100
ship = ship.sort_values('Margin')
colors_ship = ['#4FA37B' if m>=overall_margin else '#C9564A' for m in ship['Margin']]

fig, ax = plt.subplots(figsize=(8,5))
bars = ax.barh(ship.index, ship['Margin'], color=colors_ship)
ax.axvline(overall_margin, color='#555555', linewidth=1, linestyle='--')
for b, m in zip(bars, ship['Margin']):
    ax.text(m+0.15, b.get_y()+b.get_height()/2, f'{m:.1f}%', va='center', fontsize=10, fontweight='bold')
ax.set_title('Segmentasi Channel: Margin Profit per Ship Mode\n(Hijau = di atas rata-rata, Merah = di bawah rata-rata)', fontsize=12, fontweight='bold')
ax.set_xlabel('Margin Profit (%)')
plt.tight_layout()
plt.show()

print(ship)

**Insight 10:** First Class dan Second Class (hijau) memiliki margin di atas rata-rata perusahaan. Standard Class (merah) - meski volumenya paling besar (5.968 order) dan profit absolutnya tertinggi ($164k) - justru punya margin paling rendah (12,1%), menandakan efisiensi channel ini masih bisa ditingkatkan.

## 6. Tiga Temuan Utama (Key Findings)

1. **Tables adalah sumber kerugian terbesar** (-$17.725, margin -8,6%) dari seluruh 17 sub-kategori produk - jauh melampaui kandidat masalah lain seperti Bookcases (-$3.473).

2. **Diskon berkorelasi negatif dengan profit** (r=-0,22) - rata-rata profit positif saat diskon di bawah 20%, lalu berbalik negatif dan terus memburuk begitu diskon melewati 20%, terutama terlihat jelas pada bar chart Discount vs Profit.

3. **Ada trade-off antara volume dan margin** di level Segment maupun Region - Consumer/Central Region (merah) punya volume besar tapi margin di bawah rata-rata perusahaan, sementara Home Office/West Region (hijau) lebih kecil volumenya tapi lebih efisien profitnya.

Ketiga temuan ini menjadi dasar untuk **Business Analysis & Problem Solving** pada Hari Kamis.